# PneumoScan CNN: Chest X-Ray Pneumonia Detection

## Objective

This project trains a **CNN Deep Learning model** to classify chest X-ray images into:

| Class | Meaning |
|---|---|
| NORMAL | Chest X-ray appears normal |
| PNEUMONIA | Chest X-ray indicates pneumonia pattern |

> **Note:** This project is only for learning and portfolio demonstration. It is not for real medical diagnosis.

---

## Complete Project Flow

```text
1. Install/import libraries
2. Set dataset path
3. Check train / val / test folders
4. Load image data
5. Apply preprocessing and augmentation
6. Build CNN model
7. Compile model
8. Train model
9. Evaluate model
10. Generate confusion matrix and classification report
11. Save model as .keras
12. Load saved model again
13. Test one NORMAL image and one PNEUMONIA image
14. Create Streamlit app file
15. Run app
```

## Step 1: Install Required Libraries

Run this cell only once if libraries are missing.

In [ ]:
# Run only if required
# !python -m pip install tensorflow matplotlib numpy scikit-learn pillow streamlit

## Step 2: Import Libraries

In [ ]:
import os
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

import tensorflow as tf
from tensorflow.keras.models import Sequential, load_model
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout, BatchNormalization
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from tensorflow.keras.preprocessing import image

from sklearn.metrics import classification_report, confusion_matrix

print("Libraries imported successfully")
print("TensorFlow version:", tf.__version__)

## Step 3: Set Project, Dataset, and Model Paths

Update `PROJECT_PATH` and `DATASET_PATH` according to your laptop.

Your dataset should look like this:

```text
chest_xray/
├── train/
│   ├── NORMAL/
│   └── PNEUMONIA/
├── val/
│   ├── NORMAL/
│   └── PNEUMONIA/
└── test/
    ├── NORMAL/
    └── PNEUMONIA/
```

In [ ]:
PROJECT_PATH = r"E:\Rahul Verma\document\D drive\PROJECTS\Vscode\Deeplearning"

# Change this if your dataset is directly inside DeepLearning instead of Deeplearning
DATASET_PATH = r"E:\Rahul Verma\document\D drive\PROJECTS\Vscode\DeepLearning\chest_xray"

TRAIN_DIR = os.path.join(DATASET_PATH, "train")
VAL_DIR = os.path.join(DATASET_PATH, "val")
TEST_DIR = os.path.join(DATASET_PATH, "test")

MODEL_PATH = os.path.join(PROJECT_PATH, "pneumonia_cnn_model.keras")
BEST_MODEL_PATH = os.path.join(PROJECT_PATH, "best_pneumonia_model.keras")
APP_PATH = os.path.join(PROJECT_PATH, "app.py")

print("Project path:", PROJECT_PATH)
print("Dataset path:", DATASET_PATH)
print("Train path:", TRAIN_DIR)
print("Validation path:", VAL_DIR)
print("Test path:", TEST_DIR)
print("Model save path:", MODEL_PATH)

## Step 4: Check Dataset Folder Structure

In [ ]:
required_dirs = {
    "Train": TRAIN_DIR,
    "Validation": VAL_DIR,
    "Test": TEST_DIR
}

for name, folder in required_dirs.items():
    print(f"\n{name} folder:")
    if not os.path.exists(folder):
        print("Missing:", folder)
    else:
        print("Found:", folder)
        print("Subfolders:", os.listdir(folder))

        normal_path = os.path.join(folder, "NORMAL")
        pneumonia_path = os.path.join(folder, "PNEUMONIA")

        if os.path.exists(normal_path):
            print("NORMAL images:", len(os.listdir(normal_path)))
        else:
            print("NORMAL folder missing")

        if os.path.exists(pneumonia_path):
            print("PNEUMONIA images:", len(os.listdir(pneumonia_path)))
        else:
            print("PNEUMONIA folder missing")

## Step 5: Image Settings

In [ ]:
IMAGE_SIZE = (150, 150)
BATCH_SIZE = 32

print("Image size:", IMAGE_SIZE)
print("Batch size:", BATCH_SIZE)

## Step 6: Data Augmentation and Preprocessing

In [ ]:
train_datagen = ImageDataGenerator(
    rescale=1.0 / 255,
    rotation_range=15,
    zoom_range=0.2,
    width_shift_range=0.1,
    height_shift_range=0.1,
    horizontal_flip=True
)

val_test_datagen = ImageDataGenerator(
    rescale=1.0 / 255
)

## Step 7: Load Train, Validation, and Test Images

In [ ]:
train_data = train_datagen.flow_from_directory(
    TRAIN_DIR,
    target_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="binary"
)

val_data = val_test_datagen.flow_from_directory(
    VAL_DIR,
    target_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="binary"
)

test_data = val_test_datagen.flow_from_directory(
    TEST_DIR,
    target_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="binary",
    shuffle=False
)

print("Class labels:", train_data.class_indices)
print("Use these variable names for training: train_data, val_data, test_data")

## Step 8: Show Sample Training Images

In [ ]:
images, labels = next(train_data)

plt.figure(figsize=(10, 6))

for i in range(min(6, len(images))):
    plt.subplot(2, 3, i + 1)
    plt.imshow(images[i])
    label = "PNEUMONIA" if labels[i] == 1 else "NORMAL"
    plt.title(label)
    plt.axis("off")

plt.tight_layout()
plt.show()

## Step 9: Build CNN Model

In [ ]:
model = Sequential([
    Conv2D(32, (3, 3), activation="relu", input_shape=(150, 150, 3)),
    BatchNormalization(),
    MaxPooling2D(2, 2),

    Conv2D(64, (3, 3), activation="relu"),
    BatchNormalization(),
    MaxPooling2D(2, 2),

    Conv2D(128, (3, 3), activation="relu"),
    BatchNormalization(),
    MaxPooling2D(2, 2),

    Flatten(),

    Dense(128, activation="relu"),
    Dropout(0.5),

    Dense(1, activation="sigmoid")
])

model.summary()

## Step 10: Compile Model

In [ ]:
model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

print("Model compiled successfully")

## Step 11: Add Callbacks

In [ ]:
early_stop = EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True
)

checkpoint = ModelCheckpoint(
    BEST_MODEL_PATH,
    monitor="val_accuracy",
    save_best_only=True,
    mode="max"
)

print("Callbacks created")
print("Best model will be saved at:", BEST_MODEL_PATH)

## Step 12: Train Model

Do not write `model.fit(...)`. Use actual data variables as shown below.

In [ ]:
history = model.fit(
    train_data,
    validation_data=val_data,
    epochs=10,
    callbacks=[early_stop, checkpoint]
)

## Step 13: Evaluate Model on Test Data

In [ ]:
test_loss, test_accuracy = model.evaluate(test_data)

print("Test Loss:", test_loss)
print("Test Accuracy:", test_accuracy)

## Step 14: Predict on Full Test Data

In [ ]:
y_pred_prob = model.predict(test_data)
y_pred = (y_pred_prob > 0.5).astype(int).reshape(-1)

y_true = test_data.classes

print("Prediction completed")
print("Total test images:", len(y_true))

## Step 15: Confusion Matrix and Classification Report

In [ ]:
print("\nConfusion Matrix:")
print(confusion_matrix(y_true, y_pred))

print("\nClassification Report:")
print(classification_report(
    y_true,
    y_pred,
    target_names=["NORMAL", "PNEUMONIA"]
))

## Step 16: Confusion Matrix Visualization

In [ ]:
cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(6, 5))
plt.imshow(cm, cmap="Blues")
plt.title("Confusion Matrix")
plt.colorbar()

classes = ["NORMAL", "PNEUMONIA"]
tick_marks = np.arange(len(classes))

plt.xticks(tick_marks, classes)
plt.yticks(tick_marks, classes)

for i in range(len(classes)):
    for j in range(len(classes)):
        plt.text(j, i, cm[i, j], ha="center", va="center", color="black")

plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()

## Step 17: Accuracy Graph

In [ ]:
plt.plot(history.history["accuracy"], label="Training Accuracy")
plt.plot(history.history["val_accuracy"], label="Validation Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Training vs Validation Accuracy")
plt.legend()
plt.show()

## Step 18: Loss Graph

In [ ]:
plt.plot(history.history["loss"], label="Training Loss")
plt.plot(history.history["val_loss"], label="Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training vs Validation Loss")
plt.legend()
plt.show()

## Step 19: Save Final Model

This creates the model file required by the Streamlit app.

In [ ]:
# Create project folder if it does not exist
os.makedirs(PROJECT_PATH, exist_ok=True)

model.save(MODEL_PATH)

print("Model saved successfully")
print("Saved model path:", MODEL_PATH)
print("Model file exists:", os.path.exists(MODEL_PATH))

## Step 20: Load Saved Model Again

In [ ]:
loaded_model = load_model(MODEL_PATH)

print("Saved model loaded successfully")

## Step 21: Pick One NORMAL and One PNEUMONIA Image Automatically

In [ ]:
def get_first_image(folder_path):
    for file in os.listdir(folder_path):
        if file.lower().endswith((".jpg", ".jpeg", ".png")):
            return os.path.join(folder_path, file)
    return None

normal_image_path = get_first_image(os.path.join(TEST_DIR, "NORMAL"))
pneumonia_image_path = get_first_image(os.path.join(TEST_DIR, "PNEUMONIA"))

print("Normal image:", normal_image_path)
print("Pneumonia image:", pneumonia_image_path)

## Step 22: Predict One Image Function

In [ ]:
def predict_single_image(image_path, actual_label):
    if image_path is None or not os.path.exists(image_path):
        print("Image not found for:", actual_label)
        return

    img = image.load_img(image_path, target_size=IMAGE_SIZE)

    img_array = image.img_to_array(img)
    img_array = img_array / 255.0
    img_array = np.expand_dims(img_array, axis=0)

    prediction = loaded_model.predict(img_array)
    score = float(prediction[0][0])

    predicted_label = "PNEUMONIA" if score > 0.5 else "NORMAL"

    plt.imshow(img)
    plt.axis("off")
    plt.title(f"Actual: {actual_label} | Predicted: {predicted_label}")
    plt.show()

    print("Image Path:", image_path)
    print("Actual Label:", actual_label)
    print("Predicted Label:", predicted_label)
    print("Raw Confidence Score:", score)
    print("-" * 60)

## Step 23: Test One NORMAL and One PNEUMONIA Image

In [ ]:
predict_single_image(normal_image_path, "NORMAL")
predict_single_image(pneumonia_image_path, "PNEUMONIA")

## Step 24: Create Streamlit App File Automatically

This cell creates `app.py` in your project folder.

In [ ]:
app_code = r'''import os
from pathlib import Path

import numpy as np
import streamlit as st
from PIL import Image

try:
    import tensorflow as tf
except ModuleNotFoundError:
    st.error(
        "TensorFlow is not installed. Run this command in terminal:\n\n"
        "python -m pip install tensorflow"
    )
    st.stop()

st.set_page_config(
    page_title="PneumoScan CNN",
    page_icon="🫁",
    layout="centered"
)

st.title("🫁 PneumoScan: Chest X-Ray Pneumonia Detection")
st.write("Upload a chest X-ray image and the CNN model will predict NORMAL or PNEUMONIA.")

st.warning(
    "This app is for learning and project demonstration only. "
    "It should not be used for real medical diagnosis."
)

def find_model_file():
    current_folder = Path(__file__).parent
    possible_names = [
        "pneumonia_cnn_model.keras",
        "pneumonia_cnn_model.h5",
        "best_pneumonia_model.keras",
        "best_pneumonia_model.h5",
        "model.keras",
        "model.h5",
    ]

    for name in possible_names:
        model_path = current_folder / name
        if model_path.exists():
            return str(model_path)

    for ext in ["*.keras", "*.h5"]:
        files = list(current_folder.glob(ext))
        if files:
            return str(files[0])

    return ""

st.sidebar.header("Model Settings")

auto_model_path = find_model_file()

model_path = st.sidebar.text_input(
    "Model file path",
    value=auto_model_path,
    help="Paste full model path here if model is not detected automatically."
)

image_size = st.sidebar.selectbox(
    "Image size used during training",
    options=[150, 224],
    index=0
)

threshold = st.sidebar.slider(
    "Prediction threshold",
    min_value=0.10,
    max_value=0.90,
    value=0.50,
    step=0.05
)

IMAGE_SIZE = (image_size, image_size)

@st.cache_resource
def load_cnn_model(path):
    return tf.keras.models.load_model(path)

if not model_path:
    st.error(
        "Model file not found automatically.\n\n"
        "Please keep the trained model in the same folder as app.py.\n\n"
        "Recommended name: pneumonia_cnn_model.keras"
    )
    st.stop()

if not os.path.exists(model_path):
    st.error(f"Model file not found:\n\n{model_path}")
    st.stop()

try:
    model = load_cnn_model(model_path)
    st.success(f"Model loaded successfully: {model_path}")
except Exception as e:
    st.error("Model file found, but could not be loaded.")
    st.code(str(e))
    st.stop()

def preprocess_image(uploaded_image):
    img = Image.open(uploaded_image).convert("RGB")
    img_resized = img.resize(IMAGE_SIZE)

    img_array = np.array(img_resized)
    img_array = img_array / 255.0
    img_array = np.expand_dims(img_array, axis=0)

    return img, img_array

uploaded_file = st.file_uploader(
    "Upload Chest X-Ray Image",
    type=["jpg", "jpeg", "png"]
)

if uploaded_file is not None:
    original_img, img_array = preprocess_image(uploaded_file)

    st.subheader("Uploaded X-Ray Image")
    st.image(original_img, caption="Uploaded Image", use_container_width=True)

    if st.button("Predict"):
        prediction = model.predict(img_array)
        raw_score = float(prediction[0][0])

        if raw_score > threshold:
            predicted_label = "PNEUMONIA"
            confidence = raw_score
        else:
            predicted_label = "NORMAL"
            confidence = 1 - raw_score

        st.subheader("Prediction Result")

        if predicted_label == "PNEUMONIA":
            st.error(f"Prediction: {predicted_label}")
        else:
            st.success(f"Prediction: {predicted_label}")

        st.write(f"Confidence Score: **{confidence:.2%}**")
        st.write(f"Raw Model Score: `{raw_score:.6f}`")
        st.progress(min(max(confidence, 0.0), 1.0))

        st.info(
            "Prediction logic: If raw model score is greater than threshold, "
            "prediction is PNEUMONIA; otherwise prediction is NORMAL."
        )
else:
    st.info("Please upload a chest X-ray image to start prediction.")

st.markdown("---")
st.caption("Project: Chest X-Ray Pneumonia Detection using CNN | TensorFlow + Streamlit")
'''

with open(APP_PATH, "w", encoding="utf-8") as f:
    f.write(app_code)

print("Streamlit app created successfully")
print("App path:", APP_PATH)

## Step 25: Run Streamlit App

Open VS Code terminal and run:

In [ ]:
print("Run these commands in VS Code terminal:")
print(f'cd "{PROJECT_PATH}"')
print("streamlit run app.py")

## Interview Explanation

You can explain this project like this:

> I built a deep learning project for pneumonia detection from chest X-ray images.  
> I used a CNN model to classify images into NORMAL and PNEUMONIA.  
> I used preprocessing, augmentation, convolution layers, max pooling, dropout, and sigmoid output.  
> I evaluated the model using accuracy, confusion matrix, precision, recall, and F1-score.  
> I saved the trained model and created a Streamlit app where users can upload an X-ray image and get prediction.

---

## Important Notes

1. Use `test` folder images for final checking.
2. Do not use `model.fit(...)`; replace dots with actual train data.
3. Your notebook variable names are `train_data`, `val_data`, and `test_data`.
4. Save the model before running Streamlit app.
5. Keep `app.py` and `pneumonia_cnn_model.keras` in the same folder.